# SELENE: Sea Level Near-real time Quality Control Processing

This notebook demonstrates how to use SELENE for quality control of tide gauge data.
Based on the Design & User's Guide v1.0 by Puertos del Estado.


## 1. Environment Setup

In [ ]:
# install selene with conda, to be used in scripts below. 
# we will use the direct path to it's python (/opt/conda/envs/selene_training/bin/python) 
!chmod +x ./setup.sh
!./setup.sh

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import configparser
import os
import sys
import logging
from datetime import datetime
import warnings

print("✓ Required libraries imported successfully")

## 2. Configuration Setup

### 2.1 Set up directory structure

In [ ]:
# Define base paths (modify these according to your installation)
BASE_PATH = "./"  # Update this!
DATAFILES_PATH = os.path.join(BASE_PATH, "datafiles")
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs")
CONFIG_PATH = os.path.join(BASE_PATH, "configuration")

# Create directories if they don't exist
os.makedirs(DATAFILES_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CONFIG_PATH, exist_ok=True)

print(f"Base path: {BASE_PATH}")
print(f"Datafiles path: {DATAFILES_PATH}")
print(f"Output path: {OUTPUT_PATH}")

### 2.2 Configure config.ini

In [ ]:
# Create or update config.ini
config_content = """[general]
outputfolder = output/

[foreman]
program = foreman/predicc.e
harmonicsfolder = configuration/harmonics/

[interpolation]
maxgapinminutes = 25
subsamplingmethod = first

[filterhandler]
cutoff = 0.00013889
"""

config_path = os.path.join(CONFIG_PATH, "config.ini")
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"✓ Created config.ini at: {config_path}")

## 3. Prepare Input Data

### 3.1 Data Format Requirements

SELENE expects ASCII .data files with the following columns:
1. Station identifier (7-digit code)
2. Date (YYYYMMDD)
3. Time (HHMMSS)
4. Sea level (mm)
5. Quality index (optional, Copernicus conventions)

Alternative formats supported:
- Date/Time as YYYY-MM-DD HH:MM:SS
- Without QC column

### 3.2 Create Sample Data File

In [ ]:
# Create sample data file for demonstration
sample_data = """1000001 20251230 213500 3896 1
1000001 20251230 223500 3912 1
1000001 20251230 233500 3928 1
1000001 20251231 003500 3945 1
1000001 20251231 013500 3961 1
1000001 20251231 023500 3978 1
1000001 20251231 033500 3994 1
1000001 20251231 043500 4010 1
1000001 20251231 053500 4027 1
1000001 20251231 063500 4043 1
"""

data_file_path = os.path.join(DATAFILES_PATH, "1000001.data")
with open(data_file_path, 'w') as f:
    f.write(sample_data)

print(f"✓ Created sample data file: {data_file_path}")
print("\nSample data preview:")
print(sample_data)

## 4. Running SELENE

### 4.1 Execute SELENE Processing

In [ ]:
# Target station ID
TARGET_STATION = "2059388"

# Change to SELENE directory
os.chdir(BASE_PATH)

# Run SELENE processing
print(f"Running SELENE for station {TARGET_STATION}...")
print("-" * 50)

# Execute the command
try:
    # Using subprocess to run the command
    import subprocess
    
    result = subprocess.run(
        ["/opt/conda/envs/selene_training/bin/python", "selene.py", TARGET_STATION],
        capture_output=True,
        text=True,
        timeout=300  # 5 minute timeout
    )
    
    print("STDOUT:")
    print(result.stdout)
    
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    
    if result.returncode == 0:
        print("\n✓ SELENE processing completed successfully!")
    else:
        print(f"\n⚠ SELENE exited with code {result.returncode}")
        
except subprocess.TimeoutExpired:
    print("⚠ Processing timed out after 5 minutes")
except FileNotFoundError:
    print("⚠ SELENE not found. Make sure you're in the correct directory.")
    print(f"Current directory: {os.getcwd()}")
except Exception as e:
    print(f"⚠ Error running SELENE: {str(e)}")

## 5. Buddy Check Processing

### 5.1 Run Buddy Check

In [ ]:
# Buddy check requires target year as last year of data
TARGET_YEAR = 2025

print(f"Running buddy check for station {TARGET_STATION}, year {TARGET_YEAR}...")

try:
    result = subprocess.run(
        ["/opt/conda/envs/selene_training/bin/python", "buddy_check.py", 
         TARGET_STATION, str(TARGET_YEAR)],
        capture_output=True,
        text=True,
        timeout=300
    )
    
    print("STDOUT:")
    print(result.stdout)
    
    if result.returncode == 0:
        print("\n✓ Buddy check completed successfully!")
    else:
        print(f"\n⚠ Buddy check exited with code {result.returncode}")
        
except Exception as e:
    print(f"⚠ Error running buddy check: {str(e)}")

## 6. Visualization

### 6.1 Visualize Results

In [ ]:
# Run SELENEVIS for visualization
print(f"Running SELENEVIS for station {TARGET_STATION}...")

try:
    result = subprocess.run(
        ["/opt/conda/envs/selene_training/bin/python", "selenevis.py", 
         "-station", TARGET_STATION],
        capture_output=True,
        text=True,
        timeout=300
    )
    
    print("STDOUT:")
    print(result.stdout)
    
    if result.returncode == 0:
        print("\n✓ Visualization completed!")
        print("Check the 'output' directory for generated PNG files")
    else:
        print(f"\n⚠ Visualization exited with code {result.returncode}")
        
except Exception as e:
    print(f"⚠ Error running visualization: {str(e)}")

### 6.2 View Output Files

In [ ]:
# List output files
print( OUTPUT_PATH)
output_files = []
if os.path.exists('./' + OUTPUT_PATH):
    output_files = os.listdir('./' + OUTPUT_PATH)
    
print(f"Output directory contents ({len(output_files)} files):")
for file in sorted(output_files):
    filepath = os.path.join(OUTPUT_PATH, file)
    size = os.path.getsize(filepath)
    print(f"  - {file} ({size:,} bytes)")

### 6.3 Plot Output Data

In [ ]:
#### Function to read and plot output files
def plot_output_file(filepath):
    """Read and plot SELENE output file"""
    try:
        # Read the output file
        df = pd.read_csv(filepath, sep=' ', header=None, 
                        names=['datetime', 'value', 'flag', 'flag2'])
        df['datetime'] = pd.to_datetime(df['datetime'])
        df.set_index('datetime', inplace=True)

        # Plot
        plt.figure(figsize=(12, 6))
        plt.plot(df['value'], marker='o', linewidth=2)
        plt.title(f"Sea Level Data: {filepath}")
        plt.xlabel("Date/Time")
        plt.ylabel("Sea Level (mm)")
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error plotting {filepath}: {str(e)}")

# Plot first output file if available
if output_files:
    first_file = os.path.join(OUTPUT_PATH, output_files[0])
    print(f"\nPlotting: {output_files[0]}")
    plot_output_file(first_file)

## 7. Custom Data Integration

### 7.1 Add Your Own Station

To add your own station data:

1. **Prepare data file**: Create ASCII .data file in `datafiles/`
2. **Update stations.json**: Add station configuration
3. **Update code_guide.csv** (if using): Add station metadata
4. **Run SELENE**: Execute with your station ID

In [ ]:
# Example: Adding a custom station
def add_custom_station(station_id, name, lat, lon, data_file):
    """
    Add a custom station to SELENE configuration
    
    Parameters:
    station_id (str): 7-digit station code
    name (str): Station name
    lat (float): Latitude
    lon (float): Longitude
    data_file (str): Path to data file
    """
    
    # Load existing configuration
    with open(stations_path, 'r') as f:
        stations = json.load(f)
    
    # Add new station
    stations[station_id] = {
        "name": name,
        "shortname": name[:4].upper(),
        "latitude": lat,
        "longitude": lon,
        "seriesfile": f"datafiles/{data_file}",
        "seriesseparator": " ",
        "seriesdatecolumns": "1,2,3",
        "seriesdateformat": "%Y%m%d%H%M%S",
        "seriesvaluecolumn": 4,
        "seriesqccolumn": 5,
        "qc_level_nsigma": 4,
        "qc_level_winsize": 200,
        "qc_level_splinedegree": 2,
        "qc_surge_nsigma": 5,
        "qc_surge_winsize": 1000,
        "qc_surge_splinedegree": 3,
        "qc_stucklimit": 5,
        "foremanharmfile": ""
    }
    
    # Save updated configuration
    with open(stations_path, 'w') as f:
        json.dump(stations, f, indent=4)
    
    print(f"✓ Added station {station_id} ({name}) to configuration")
    print(f"  Location: {lat}, {lon}")
    print(f"  Data file: {data_file}")

# Example usage (uncomment and modify):
# add_custom_station("2000001", "NEW_STATION", 45.5, -5.5, "2000001.data")

## 8. Troubleshooting

### Common Issues

**1. Module not found errors**
- Ensure you're in the SELENE directory
- Check that Python environment is activated: `conda activate selene_training`

**2. Permission denied**
- Give execution permissions: `chmod 755 foreman/predicc.e`

**3. Data format errors**
- Verify .data file format matches specifications
- Check column separators and date/time formats

**4. Missing harmonic constants**
- Set `foremanharmfile` to empty string ("") if not available
- Tide/surge analysis will be skipped

### Runtime Warnings

You may see warnings like:
```
RankWarning: Polyfit may be poorly conditioned
```
These are expected and don't prevent processing from completing.

## 9. Output Files Reference

SELENE generates several output files in the `output/` directory:

| Algorithm | Output Files | Description |
|-----------|--------------|-------------|
| Selene.py | `{code}_original_sampling_flags.out` | Original data with quality flags |
| Buddy_check.py | `{code}_original_sampling_buddy.out` | Buddy check results |
| | `{code}_hourly_slev_buddy.out` | Hourly sea levels |
| | `{code}_hourly_surge_buddy.out` | Hourly surges |
| | `{code}_hourly_tide_buddy.out` | Hourly tides |
| Selenevis.py | `{station}_*.png` | Visualization plots |

## 10. Next Steps

### Additional Resources

- **Official Documentation**: https://puertos-del-estado-medio-fisico.github.io/SELENE/
- **Copernicus Marine Service**: https://data.marine.copernicus.eu/product/INSITU_GLO_PHY_SSH_DISCRETE_MY_013_053/description
- **Support Contact**: Dr. Begoña Pérez Gómez - bego@puertos.es

### Advanced Usage

- **Automated Processing**: Set up cron jobs with `selenelauncher.py`
- **Database Integration**: Use `selenedatadownload.py` and `selenedbconsolide.py`
- **Custom Filters**: Modify `filterhandler.py` for different filtering methods
- **Parallel Processing**: Configure multiprocessing in launcher script

---

**Note**: This notebook is based on SELENE Version 1.0 by Puertos del Estado.
For production use, ensure all configuration parameters are properly set for your
specific stations and data characteristics.

In [ ]:
print("\n" + "="*60)
print("Notebook completed!")
print("="*60)
print("\nTo run SELENE with your data:")
print(f"  1. Place your .data file in: {DATAFILES_PATH}")
print(f"  2. Update stations.json with your station configuration")
print(f"  3. Run: /opt/conda/envs/selene_training/bin/python selene.py YOUR_STATION_ID")
print(f"  4. Visualize: /opt/conda/envs/selene_training/bin/python selenevis.py -station YOUR_STATION_ID")